# Modeling (Im)Precision in Context

This notebook accompanies the case study in:

> Mühlenbernd, R., Solt, S., & Sauerland, U. (in press). *[Chapter title]*. In *[Edited Volume]*.

It reproduces the empirical analysis and probabilistic speaker model from:

> Mühlenbernd, R. & Solt, S. (2022). Modeling (im)precision in context. *Linguistics Vanguard*, 8(1), 113–127. https://doi.org/10.1515/lingvan-2022-0035

---

## Overview

How precisely do speakers communicate numerical information — and does context matter?

In Mühlenbernd & Solt (2022), we report a **production experiment** in which 499 participants reported the time of a witnessed accident. The key manipulations were **context** (police station vs. neighbor at a party) and **information state** (how precisely the participant knew the time). The central finding: speakers round off more in the neighbor context than in the police context. A probabilistic game-theoretic speaker model reproduces this pattern by fitting context-specific weights for the speaker goals of accuracy and hearer-oriented simplification.

**Notebook structure:**
1. Environment setup (local and Colab)
2. Data loading and cleaning
3. *(Optional)* Descriptive statistics
4. *(Optional)* Response matrices
5. *(Optional)* Chi-square tests of context effects
6. *(Optional)* Motive analysis
7. Probabilistic speaker model *(coming soon)*

Sections marked *optional* reproduce the empirical analysis from the paper. They can be skipped if you are primarily interested in the model.

## 1. Environment Setup

The cell below detects whether the notebook is running locally or in Google Colab.
In Colab, it clones the repository so that the helper modules in `src/` are available for import.

In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    if not os.path.exists('imprecision-in-context'):
        !git clone https://github.com/muehlenbernd/imprecision-in-context.git
    os.chdir('imprecision-in-context/notebooks')
    print('Running in Colab — repository cloned.')
else:
    print('Running locally.')

# Add src/ to the path so helper modules can be imported
sys.path.insert(0, os.path.abspath('../src'))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency

# Helper modules (outsourced to src/)
from stats.analysis import (
    build_matrix, plot_response_matrices, plot_motives,
    chi2_context_test, chi2_per_state,
    STATE_LABELS, STATE_ORDER
)

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 12,
    'figure.dpi': 120
})

print('Imports OK.')

## 2. Data Loading and Cleaning

The participant-level data is available at:
- Figshare: https://doi.org/10.6084/m9.figshare.21629531
- This repository: `data/raw/Participant_level_data_1_.xlsx`

The Excel file contains two sheets:
- `FillerStudyFullClass`: 499 participant-level rows
- `Description`: codebook for all columns

### Column guide

| Column | Description |
|--------|-------------|
| `context` | `police` or `neighbor` |
| `stateC` | Information state category (0–5, 8) |
| `answerC` | Response category; `-1` = excluded (miscoded clock) |
| `appxC` | Approximator term used (e.g. *around*, *about*; `0` = none) |
| `Precision`…`Other` | Binary motive flags |
| `motive` | Free-text justification |

### Information state coding

| `stateC` | Information state | Interpretation |
|----------|-------------------|----------------|
| 0 | 8:30 | Exact round value |
| 1 | 8:29 or 8:31 | ±1 minute off round |
| 2 | 8:28 or 8:32 | ±2 minutes off round |
| 3 | 8:27 or 8:33 | ±3 minutes off round |
| 4 | 8:26 or 8:34 | ±4 minutes off round |
| 5 | 8:25 or 8:35 | ±5 minutes (divisible by 5) |
| 8 | 8:26–8:34 | Approximate range |

In [ ]:
GITHUB_URL = (
    'https://raw.githubusercontent.com/muehlenbernd/'
    'imprecision-in-context/main/data/raw/Participant_level_data_1_.xlsx'
)
LOCAL_PATH = '../data/raw/Participant_level_data_1_.xlsx'

data_source = LOCAL_PATH if os.path.exists(LOCAL_PATH) else GITHUB_URL
print(f'Loading from: {data_source}')

df_raw = pd.read_excel(data_source, sheet_name='FillerStudyFullClass', header=1)
print(f'Total participants loaded: {len(df_raw)}')

In [ ]:
# Exclude participants whose clock readings could not be coded (answerC == -1)
n_excluded = (df_raw['answerC'] == -1).sum()
df = df_raw[df_raw['answerC'] != -1].copy()

print(f'Excluded: {n_excluded}')
print(f'Remaining: {len(df)}')
print()
print(df['context'].value_counts().to_string())

## 3. Descriptive Statistics *(optional)*

Participant counts per condition (context × information state).

In [ ]:
counts = df.groupby(['context', 'stateC']).size().unstack()
counts.index.name = 'Context'
counts.columns = [STATE_LABELS[c] for c in counts.columns]
counts

## 4. Response Matrices *(optional)*

Each matrix has rows = information states, columns = response categories, cells = proportion of participants choosing that response. These are the normalized matrices shown in **Figure 2** of the paper.

Response categories are collapsed to match the paper: all interval expressions (bare intervals and approximator + interval variants) are merged into a single *v*ᵢₙ column. Reading across a row shows how participants distributed their choices given a particular clock time; the key comparison is police vs. neighbor — more concentration in the *v*₃₀ (bare 8:30) column in the neighbor context indicates more rounding.

In [ ]:
matrix_police   = build_matrix(df, 'police')
matrix_neighbor = build_matrix(df, 'neighbor')

plot_response_matrices(
    matrix_police, matrix_neighbor,
    title='Normalized response matrices (Figure 2, Mühlenbernd & Solt 2022)',
    save_path='../figures/figure2_response_matrices.png'
)

## 5. Chi-Square Tests of Context Effects *(optional)*

We test whether the distribution of responses differs significantly between contexts. The paper reports two key tests:
- Non-round precise states (s₃₀±₁ through s₃₀±₄) pooled: χ²=13.198, *p*<0.001
- Approximate state (sᵢₙ) alone: χ²=5.121, *p*=0.077 (near-significant)

The per-state breakdown below additionally shows where the context effect is strongest.

In [ ]:
# Pooled test: non-round precise states
chi2, p, dof = chi2_context_test(df, states=[1, 2, 3, 4])
print(f'Non-round precise states (s₃₀±₁ to s₃₀±₄):')
print(f'  χ²({dof}) = {chi2:.3f}, p = {p:.4f}')
print(f'  Paper reports: χ²=13.198, p<0.001')
print()

chi2, p, dof = chi2_context_test(df, states=[8])
print(f'Approximate state (sᵢₙ):')
print(f'  χ²({dof}) = {chi2:.3f}, p = {p:.4f}')
print(f'  Paper reports: χ²=5.121, p=0.077')

In [ ]:
# Per-state breakdown
chi2_per_state(df)

## 6. Motive Analysis *(optional)*

After answering the main question, participants were asked to justify their choice. Responses were coded into 11 categories (plus *other*); multiple categories could apply.

The motive profiles differ systematically by context: accuracy concerns dominate in the police context, while ease and habit dominate in the neighbor context. This provides independent behavioral evidence for the model's core assumption — that speakers weight the goals of accuracy (*w*ₐ) and hearer-oriented simplification (*w*ᵣ) differently depending on the situation.

It also raises an interesting question for LLM evaluation: when prompted to justify a precision choice, do language models produce motive profiles that resemble human speakers?

In [ ]:
motive_df = plot_motives(df, save_path='../figures/figure_motives.png')
motive_df

## 7. Probabilistic Speaker Model

*This section is under construction and will be completed in the next iteration.*

The model to be implemented here is the Imprecision Model from Mühlenbernd & Solt (2022). It predicts the probability that a speaker uses utterance *v* given information state *s* by defining a total utility:

$$U_{tot}(v, s, w) = U_{inf}(v, s) + w_R \cdot U_{rnd}(v) + w_A \cdot U_{acc}(v, s) - C(v)$$

where:
- $U_{inf}$: informativity utility — how well *v* conveys *s* to the hearer
- $U_{rnd}$: roundness utility — reward for hearer-friendly round numbers (weighted by $w_R$)
- $U_{acc}$: accuracy utility — reward for utterances likely to be true (weighted by $w_A$)
- $C(v)$: utterance cost — penalty for longer/more complex forms

Speaker choice follows a softmax over total utility. By fitting $w_R$ and $w_A$ separately for each context, the model reproduces the empirical matrices with Pearson *r* = 0.985.

Implementation will be added to `src/model/imprecision_model.py` and called here.

---

## Citation

If you use this data or code, please cite:

```bibtex
@article{muhlenbernd_solt_2022,
  author  = {Mühlenbernd, Roland and Solt, Stephanie},
  title   = {Modeling (im)precision in context},
  journal = {Linguistics Vanguard},
  year    = {2022},
  volume  = {8},
  number  = {1},
  pages   = {113--127},
  doi     = {10.1515/lingvan-2022-0035}
}
```